In [22]:
from dotenv import load_dotenv

load_dotenv()

False

In [23]:
import os
mlflow_uri = os.getenv("MLFLOW_TRACKING_URI")
username = os.getenv("MLFLOW_TRACKING_USERNAME")
password = os.getenv("MLFLOW_TRACKING_PASSWORD")

In [2]:
%pwd

'd:\\projects\\wine-quality-mlops\\research'

In [4]:
os.chdir("..")
%pwd

'd:\\projects\\wine-quality-mlops'

In [7]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict[str, Any]
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [9]:
from wine_quality_mlops.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from wine_quality_mlops.utils.io_utils import read_yaml, create_directories, save_json

In [20]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        self.artifacts_root = Path(self.config["artifacts_root"])

        create_directories([self.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:

        config = self.config["model_evaluation"]
        params = self.params["ElasticNet"]
        target_column = self.schema["TARGET_COLUMN"]["name"]

        root_dir = self.artifacts_root / config["root_dir"]

        create_directories([root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir= root_dir,
            test_data_path= Path(config["test_data_path"]),
            model_path= Path(config["model_path"]),
            all_params= params,
            metric_file_name= root_dir / config["metric_file_name"],
            target_column= target_column,
            mlflow_uri= "https://dagshub.com/inglesantosh09/wine-quality-mlops.mlflow"
        )

        return model_evaluation_config

In [16]:
import sys
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from wine_quality_mlops.utils.exceptions import CustomException
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [12]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self,actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]


        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            
            # Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)


            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [21]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise CustomException(e, sys)

[2026-05-17 14:25:18,507] io_utils.py:22 datascienceLogger - INFO - YAML file loaded: D:\projects\wine-quality-mlops\config\config.yaml
[2026-05-17 14:25:18,508] io_utils.py:22 datascienceLogger - INFO - YAML file loaded: D:\projects\wine-quality-mlops\params.yaml
[2026-05-17 14:25:18,510] io_utils.py:22 datascienceLogger - INFO - YAML file loaded: D:\projects\wine-quality-mlops\schema.yaml
[2026-05-17 14:25:18,511] io_utils.py:40 datascienceLogger - INFO - Created directory: artifacts
[2026-05-17 14:25:18,512] io_utils.py:40 datascienceLogger - INFO - Created directory: artifacts\model_evaluation
[2026-05-17 14:25:21,007] io_utils.py:51 datascienceLogger - INFO - JSON file saved: artifacts\model_evaluation\metrics.json


2026/05/17 14:25:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 14:25:22 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\projects\wine-quality-mlops
2026/05/17 14:25:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/17 14:25:25 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\projects\wine-quality-mlops
2026/05/17 14:25:25 INFO mlflow.utils.environment: Detected uv project at d:\projects\wine-quality-mlops. Attempting to export requirements via 'uv export'.
2026/05/17 14:25:28 INFO mlflow.utils.uv_utils: Exported 169 de

🏃 View run trusting-mouse-67 at: https://dagshub.com/inglesantosh09/wine-quality-mlops.mlflow/#/experiments/0/runs/eab4ddbec8d6419997b2a9c0aa068ac3
🧪 View experiment at: https://dagshub.com/inglesantosh09/wine-quality-mlops.mlflow/#/experiments/0
